# Notebook 04: Forecasting BREAD/BAKERY demand by warehouse

**Project:** Perishable Inventory Optimizer (GIS + supply chain)
**Milestone:** 3, Demand aggregation and forecast (continued from notebook 03)

## What we are trying to accomplish
Predict how many units of BREAD/BAKERY each of the four warehouse regions will sell tomorrow, and find out whether a machine-learning model predicts better than simple rules. Everything is tested on eight weeks of data that the models never saw while they were being built.

## What we got accomplished
Nothing yet. This notebook was just created and its code has been tested, but we have not yet run it together. The steps are numbered 4 to 7 because notebook 03 holds Steps 1 to 3.

## What we have left to do
1. **Step 4:** build the BREAD/BAKERY table (2015 onward) and split it by date into train, validation and test.
2. **Step 5:** simple baseline forecasts, the bar to beat.
3. **Step 6:** a machine-learning model, tuned on the validation weeks and scored once on the test weeks.
4. **Step 7:** compare the two on the test weeks.
5. Then (Milestone 4): turn forecasts into stocking recommendations that balance waste against stockouts.

**How each step works:** we run the code, look at the result, and fill in the explanation cells together: what we are looking for, how to read it, what it means, and what is next and why.

**Rules we keep:** we never use the contest files (`test.csv`, `sample_submission.csv`); our test set is the last 56 days of `train.csv`. The test weeks are scored once, after every choice is made. A sale is not the same as demand, because a sold-out shelf hides unmet demand.

**Where the full process notes are:** `docs/forecast_process_notes.md` records every option tried, why one was chosen and the numbers, for Steps 4 to 6.

## Step 4. The BREAD/BAKERY table and the train / validation / test split

### What we are looking for
One clean row for every warehouse and open day of BREAD/BAKERY from 2015-01-01, and a fair way to test a forecast: split the days by date into three periods so a model is always judged on days that come after the ones it learned from.

### How we got here (the process)
1. **Load BREAD/BAKERY from 2015-01-01 (4a).** We start in 2015 because of the unexplained rise in 2014 (notebook 03, Step 3). We merge in the count of items on promotion and check that no day was flagged as unusual.
2. **Use units per open store as the thing to predict.** Total sales jump when a store opens or closes, which is not a change in customer demand. Total = per-store forecast x open stores.
3. **Drop warehouse-days with no open store (9 rows, all Jan 1).** Options considered: keep them as zero (teaches the model that demand vanishes on Jan 1), fill them with an estimate (invents data), or drop them. We drop them: nothing was open, so nothing needs forecasting.
4. **Split by date, in time order (4b).** Test = the last 56 days (2017-06-21 to 2017-08-15), validation = the 56 days before that, train = everything earlier. 56 days is eight whole weeks, so every weekday appears eight times in each window. A random split was rejected because it lets a model see days after the ones it is tested on, which flatters the score.
5. **Roles of the three sets.** Train: the model learns from it. Validation: we use it to choose between models and settings. Test: touched once at the end, only to report the final score.
6. **Chart the split (4c) and run checks (4d):** missing calendar days, holidays in the test window, and whether the number of open stores changes.

**Assumptions and limits:** only about 2.6 years of training data. The test window is mid-year, so it says nothing about December. The number of open stores is assumed known a day ahead (it is constant in validation and test). The saved file `bread_daily.csv` is a derived file and is not committed to GitHub.

In [23]:
# 4a. Load the BREAD/BAKERY table: one row per warehouse per open day
import warnings
from pathlib import Path
import numpy as np
import pandas as pd

PROCESSED = Path("../data/processed")

# The "open" file has sales per open store; the other file has the promotion counts.
opn = pd.read_csv(PROCESSED / "perishable_daily_open.csv", parse_dates=["date"])
raw = pd.read_csv(PROCESSED / "perishable_daily_by_warehouse.csv", parse_dates=["date"])

# Bread only, and only from 2015 on (a level jump in mid-2014 is unexplained).
FIRST_DAY = pd.Timestamp("2015-01-01")
bread_all = opn[(opn["family"] == "BREAD/BAKERY") & (opn["date"] >= FIRST_DAY)].copy()

# We were told no bread day was flagged unusual; stop here if that is ever false.
assert not bread_all["unusual"].any(), "Some BREAD/BAKERY days are flagged unusual"

# Warehouse-days where every store was closed have nothing to forecast: drop them, but list them.
no_open = bread_all[bread_all["open_stores"] == 0]
print(f"Warehouse-days dropped (no open store): {len(no_open)}")
display(no_open[["date", "warehouse"]].sort_values("date").reset_index(drop=True))

bread = bread_all[bread_all["open_stores"] > 0].copy()

# Bring in on_promotion (number of items on promotion that day, summed over the warehouse's stores).
promo = raw.loc[raw["family"] == "BREAD/BAKERY", ["date", "warehouse", "on_promotion"]]
n_before = len(bread)
bread = bread.merge(promo, on=["date", "warehouse"], how="left", validate="one_to_one")
assert len(bread) == n_before and bread["on_promotion"].notna().all(), "promotion merge lost rows"

# Target: units per OPEN store, so a new store opening does not look like a demand jump.
bread["units_per_open_store"] = bread["sales"] / bread["open_stores"]

bread = (bread[["date", "warehouse", "sales", "open_stores", "on_promotion", "units_per_open_store"]]
         .sort_values(["warehouse", "date"]).reset_index(drop=True))

print(f"Rows: {len(bread):,}   Date range: {bread['date'].min().date()} to {bread['date'].max().date()}")
summary = (bread.groupby("warehouse")["date"].agg(days="count", first="min", last="max"))
display(summary)

Warehouse-days dropped (no open store): 9


,date,warehouse
0,2015-01-01,Cuenca
1,2015-01-01,Quito
2,2015-01-01,Santo Domingo
3,2016-01-01,Cuenca
4,2016-01-01,Quito
5,2016-01-01,Santo Domingo
6,2017-01-01,Cuenca
7,2017-01-01,Quito
8,2017-01-01,Santo Domingo


Rows: 3,815   Date range: 2015-01-01 to 2017-08-15


,days,first,last
warehouse,,,
Cuenca,953,2015-01-02,2017-08-15
Guayaquil,956,2015-01-01,2017-08-15
Quito,953,2015-01-02,2017-08-15
Santo Domingo,953,2015-01-02,2017-08-15


### How to read it
(Fill in together after running.)

### What it means
(Fill in together after running.)

### What's next and why
(Fill in together after running.)

In [24]:
# 4b. Chronological split: last 56 days = test, the 56 before = validation, the rest = train
# 56 days = 8 whole weeks, so every weekday appears the same number of times in each window.
LAST_DAY = pd.Timestamp("2017-08-15")
WINDOW = 56
test_start = LAST_DAY - pd.Timedelta(days=WINDOW - 1)
val_start = test_start - pd.Timedelta(days=WINDOW)

assert bread["date"].max() == LAST_DAY, "data does not end on 2017-08-15"

bread["split"] = np.select(
    [bread["date"] >= test_start, bread["date"] >= val_start],
    ["test", "validation"],
    default="train",
)

order = ["train", "validation", "test"]
split_table = (bread.groupby("split")
               .agg(first_date=("date", "min"), last_date=("date", "max"),
                    days=("date", "nunique"), warehouse_days=("date", "size"))
               .loc[order])
display(split_table)

# Checks: sets are in time order and do not overlap.
firsts = split_table["first_date"].tolist()
lasts = split_table["last_date"].tolist()
assert lasts[0] < firsts[1] and lasts[1] < firsts[2], "splits overlap or are out of order"
assert split_table["warehouse_days"].sum() == len(bread)
print("OK: train < validation < test, no overlap")

bread.to_csv(PROCESSED / "bread_daily.csv", index=False)
print(f"Saved {len(bread):,} rows to {PROCESSED / 'bread_daily.csv'}")

,first_date,last_date,days,warehouse_days
split,,,,
train,2015-01-01,2017-04-25,844,3367
validation,2017-04-26,2017-06-20,56,224
test,2017-06-21,2017-08-15,56,224


OK: train < validation < test, no overlap
Saved 3,815 rows to ..\data\processed\bread_daily.csv


### How to read it
(Fill in together after running.)

### What it means
(Fill in together after running.)

### What's next and why
(Fill in together after running.)

In [25]:
# 4c. Chart: units per open store, by day, with the three periods shaded
import plotly.graph_objects as go

# One muted line colour per warehouse; the periods are shown as pale background bands.
LINE_COLOURS = {"Quito": "#1f3b5c", "Guayaquil": "#4f7a9c", "Cuenca": "#8c6d46", "Santo Domingo": "#6b8f71"}
BANDS = [("train", FIRST_DAY, val_start, "rgba(150,150,150,0.08)"),
         ("validation", val_start, test_start, "rgba(70,120,180,0.15)"),
         ("test", test_start, LAST_DAY + pd.Timedelta(days=1), "rgba(200,120,60,0.18)")]

fig = go.Figure()
for wh, grp in bread.groupby("warehouse"):
    fig.add_trace(go.Scatter(x=grp["date"], y=grp["units_per_open_store"], mode="lines",
                             name=wh, line=dict(width=1, color=LINE_COLOURS.get(wh))))
for name, x0, x1, colour in BANDS:
    fig.add_vrect(x0=x0, x1=x1, fillcolor=colour, line_width=0,
                  annotation_text=name, annotation_position="top left", layer="below")
fig.update_layout(title="BREAD/BAKERY: units per open store per day, with train / validation / test periods",
                  xaxis_title="Date", yaxis_title="Units per open store",
                  template="plotly_white", height=500, legend_title_text="Warehouse")
fig.show()

### How to read it
(Fill in together after running.)

### What it means
(Fill in together after running.)

### What's next and why
(Fill in together after running.)

In [26]:
# 4d. Checks: calendar gaps, holidays in the test window, and open-store changes
# (1) Gaps: days missing from a set. Expected absences are not gaps: days with no rows for any
# warehouse (the source data has no Dec 25) and the warehouse-days already dropped in 4a.
all_dates = pd.date_range(FIRST_DAY, LAST_DAY, freq="D")
absent_everywhere = set(all_dates.difference(bread["date"].unique()))
dropped = set(zip(no_open["warehouse"], no_open["date"]))
print(f"Calendar days with no data for any warehouse: {sorted(str(d.date()) for d in absent_everywhere)}")
unexpected = 0
for name in order:
    part = bread[bread["split"] == name]
    calendar = pd.date_range(part["date"].min(), part["date"].max(), freq="D")
    for wh, grp in part.groupby("warehouse"):
        missing = [d for d in calendar.difference(grp["date"])
                   if d not in absent_everywhere and (wh, d) not in dropped]
        unexpected += len(missing)
        if missing:
            print(f"{name:<10} {wh:<14} unexpected gap(s): {[str(d.date()) for d in missing]}")
print(f"Unexpected gaps in train/validation/test: {unexpected}")

# (2) National holidays in the test window (reported only; the modelling step will use them).
hol = pd.read_csv("../data/raw/store-sales-time-series-forecasting/holidays_events.csv", parse_dates=["date"])
# A day counts as a real national day off if it is national and not moved to another date.
nat = hol[(hol["locale"] == "National") & (~hol["transferred"]) &
          (hol["date"] >= test_start) & (hol["date"] <= LAST_DAY)]
print(f"National entries in the test window: {len(nat)}")
display(nat[["date", "type", "description"]])
jan1 = [d for d in pd.date_range(test_start, LAST_DAY) if d.month == 1 and d.day == 1]
print(f"Jan 1 inside the test window: {bool(jan1)}")

# (3) Does the number of open stores change in validation / test?
for name in ["validation", "test"]:
    part = bread[bread["split"] == name]
    changes = part.groupby("warehouse")["open_stores"].agg(["min", "max", "nunique"])
    print(f"Open stores during {name}:")
    display(changes)

Calendar days with no data for any warehouse: ['2015-12-25', '2016-12-25']
Unexpected gaps in train/validation/test: 0
National entries in the test window: 1


,date,type,description
325,2017-08-11,Transfer,Traslado Primer Grito de Independencia


Jan 1 inside the test window: False
Open stores during validation:


,min,max,nunique
warehouse,,,
Cuenca,7,7,1
Guayaquil,16,16,1
Quito,25,25,1
Santo Domingo,6,6,1


Open stores during test:


,min,max,nunique
warehouse,,,
Cuenca,7,7,1
Guayaquil,16,16,1
Quito,25,25,1
Santo Domingo,6,6,1


### How to read it
(Fill in together after running.)

### What it means
(Fill in together after running.)

### What's next and why
(Fill in together after running.)

## Step 5. Baseline forecasts: the bar to beat

### What we are looking for
The accuracy of simple, explainable forecasts. A machine-learning model only earns its place if it beats these by a clear margin.

### How we got here (the process)
1. **One-day-ahead forecasts.** Each day a warehouse decides how much bread to stock for tomorrow, so the forecast for day d may use actual data up to day d-1 and things known in advance (weekday, open stores). It never sees day d. A check in 5c plants a huge number on one day and confirms no baseline's forecast for that day changes.
2. **Three scores, in one function (5b).** WAPE is the total absolute miss divided by total sales, the headline number (5% means we miss by 5% of sales). Bias is the net lean: positive means we over-forecast (bread left over, waste), negative means we under-forecast (stockout risk). MAE is the average miss in units. Look at WAPE and bias together: a forecast can be right on average and wrong every day.
3. **Eight candidates (5c):** yesterday's value (naive); the same weekday last week (seasonal naive); the mean of the last 4 or 8 same weekdays; the median of the last 4 or 8 same weekdays (added because one odd day cannot pull a median); and the mean of 4 or 8 same weekdays times a recent-level adjustment.
4. **Choose on validation only (5d).** The rule was fixed before looking: lowest overall validation WAPE. We tried only N = 4 and 8, not a wider search, to avoid fitting the baseline to one window.
5. **Score on test, once (5e):** only the chosen baseline plus the plain seasonal naive as the well-known reference. Nothing is changed afterwards.
6. **Rule for a missing day:** a day with no value takes the value from the same weekday one week earlier, because bread has a strong weekly pattern.

**Assumptions and limits:** one 8-week validation window, so differences of about 0.1 point between candidates mean nothing. The baselines know nothing about holidays or promotions. Test WAPE and validation WAPE come from different windows and must not be compared with each other, only within the same window.

In [27]:
# 5a. Load the bread table and set up the one-day-ahead history
from pathlib import Path
import numpy as np
import pandas as pd

PROCESSED = Path("../data/processed")
bread = pd.read_csv(PROCESSED / "bread_daily.csv", parse_dates=["date"])

# WHY: every baseline looks back by whole days (yesterday, 7 days ago, ...). Putting each warehouse
# on a complete calendar (one column per warehouse) turns "d-1" and "d-7" into simple row shifts.
calendar = pd.date_range(bread["date"].min(), bread["date"].max(), freq="D")
per_store_raw = bread.pivot(index="date", columns="warehouse", values="units_per_open_store").reindex(calendar)
open_stores = bread.pivot(index="date", columns="warehouse", values="open_stores").reindex(calendar)

# MISSING-DAY RULE: if a day has no value (Jan 1 closures, Dec 25 not in the data), the history
# uses the value from the same weekday one week earlier. WHY: bread has a strong weekly pattern,
# so last week's same day is a better stand-in than "the day before" (e.g. a Sunday gap filled
# with Saturday). It only looks backwards, so it never uses information from the future.
def fill_gaps(wide):
    values = wide.to_numpy(copy=True)
    for i in range(7, len(values)):
        gap = np.isnan(values[i])
        values[i, gap] = values[i - 7, gap]
    return pd.DataFrame(values, index=wide.index, columns=wide.columns)

per_store = fill_gaps(per_store_raw)
n_gaps = int(per_store_raw.isna().sum().sum())
n_left = int(per_store.iloc[7:].isna().sum().sum())
print(f"Calendar days: {len(calendar)}   gaps filled by the same-weekday rule: {n_gaps - int(per_store_raw.iloc[:7].isna().sum().sum())}")
print(f"Gaps left after the rule (after the first 7 days): {n_left}")
assert n_left == 0

# Only real warehouse-days are scored. Validation and test must be complete (nothing to fill).
for name in ["validation", "test"]:
    part = bread[bread["split"] == name]
    assert len(part) == 224, f"{name} is not complete"
print(bread.groupby("split").size().to_frame("warehouse-days").loc[["train", "validation", "test"]])

Calendar days: 958   gaps filled by the same-weekday rule: 14
Gaps left after the rule (after the first 7 days): 0
            warehouse-days
split                     
train                 3367
validation             224
test                   224


### How to read it
(Fill in together after running.)

### What it means
(Fill in together after running.)

### What's next and why
(Fill in together after running.)

In [28]:
# 5b. One reusable score function: MAE, WAPE and bias on TOTAL units per warehouse-day
# WAPE = sum of |forecast - actual| / sum of actual   (the headline number; lower is better)
# bias = sum of (forecast - actual) / sum of actual   (positive = over-forecast = waste risk,
#                                                      negative = under-forecast = stockout risk)
def score(df, actual="actual", forecast="forecast"):
    """Return one row for all data ('All') plus one row per warehouse."""
    def metrics(g):
        err = g[forecast] - g[actual]
        total = g[actual].abs().sum()
        return {"days": len(g), "MAE": err.abs().mean(),
                "WAPE": err.abs().sum() / total, "bias": err.sum() / total}
    rows = {"All": metrics(df)}
    for wh, g in df.groupby("warehouse"):
        rows[wh] = metrics(g)
    return pd.DataFrame(rows).T.astype({"days": int})

def pretty(t):
    """Show WAPE and bias as percentages, MAE in units."""
    out = t.copy()
    out["WAPE"] = (out["WAPE"] * 100).round(2)
    out["bias"] = (out["bias"] * 100).round(2)
    out["MAE"] = out["MAE"].round(1)
    return out.rename(columns={"WAPE": "WAPE %", "bias": "bias %", "MAE": "MAE units"})

# Tiny check by hand: actual 100 and 100, forecast 110 and 90 -> errors +10 and -10.
toy = pd.DataFrame({"warehouse": ["A", "A"], "actual": [100.0, 100.0], "forecast": [110.0, 90.0]})
toy_score = score(toy).loc["All"]
assert abs(toy_score["MAE"] - 10) < 1e-9 and abs(toy_score["WAPE"] - 0.10) < 1e-9 and abs(toy_score["bias"]) < 1e-9
print("score() passes the hand check (MAE 10, WAPE 10%, bias 0%)")

score() passes the hand check (MAE 10, WAPE 10%, bias 0%)


### How to read it
(Fill in together after running.)

### What it means
(Fill in together after running.)

### What's next and why
(Fill in together after running.)

In [29]:
# 5c. Baselines, one-day-ahead, scored on VALIDATION only
# Every baseline for day d reads only days up to d-1 (lags of 1, 7, 14, ... days).
def lag_stack(F, n_weeks):
    """The same weekday in each of the last n_weeks weeks: lags 7, 14, ..., 7*n_weeks."""
    return np.stack([F.shift(7 * k).to_numpy() for k in range(1, n_weeks + 1)])

def baseline_per_store(F, model):
    """Forecast units per open store for every day, using only earlier days."""
    if model == "naive":                       # 1. yesterday
        return F.shift(1)
    if model == "seasonal_naive":              # 2. same weekday last week
        return F.shift(7)
    kind, n = model.split("_")[0].rstrip("0123456789"), int(model.split("_")[0].lstrip("abcdefghijklmnopqrstuvwxyz"))
    stack = lag_stack(F, n)
    if kind == "mean":                         # 3. mean of last n same weekdays
        base = np.mean(stack, axis=0)
    elif kind == "median":                     # added: median of last n same weekdays
        base = np.median(stack, axis=0)
    else:
        raise ValueError(model)
    base = pd.DataFrame(base, index=F.index, columns=F.columns)
    if model.endswith("_level"):               # 4. times last 7 days / the 7 days before them
        week_sum = F.rolling(7).sum()
        base = base * (week_sum.shift(1) / week_sum.shift(8))
    return base

MODELS = ["naive", "seasonal_naive", "mean4", "mean8", "median4", "median8", "mean4_level", "mean8_level"]

def forecast_table(model, split):
    """Long table for one model and one split: forecast converted to TOTAL units."""
    total = baseline_per_store(per_store, model) * open_stores
    fc = total.reset_index(names="date").melt(id_vars="date", var_name="warehouse", value_name="forecast")
    rows = bread.loc[bread["split"] == split, ["date", "warehouse", "split", "sales"]].rename(columns={"sales": "actual"})
    out = rows.merge(fc, on=["date", "warehouse"], how="left", validate="one_to_one")
    assert out["forecast"].notna().all(), f"{model}: some forecasts are missing"
    out.insert(3, "model", model)
    return out[["date", "warehouse", "split", "model", "actual", "forecast"]]

# Leakage check: put a huge number on one day and rebuild the forecasts. The forecast FOR that
# day must not move (it may only use days before it); the next day's forecast may move.
probe = pd.Timestamp("2017-05-10")
for m in MODELS:
    clean = baseline_per_store(per_store, m)
    dirty_F = per_store.copy()
    dirty_F.loc[probe] = 1e9
    dirty = baseline_per_store(dirty_F, m)
    assert np.allclose(clean.loc[probe], dirty.loc[probe], equal_nan=True), f"{m} peeks at the day it forecasts"
print("Leakage check passed: no baseline sees the day it forecasts")

val_fc = pd.concat([forecast_table(m, "validation") for m in MODELS], ignore_index=True)
val_scores = {m: score(g) for m, g in val_fc.groupby("model")}

overall = pd.DataFrame({m: s.loc["All"] for m, s in val_scores.items()}).T.loc[MODELS]
print("Validation, all warehouses together (sorted by WAPE):")
display(pretty(overall.drop(columns="days")).sort_values("WAPE %"))

wape_wh = pd.DataFrame({m: s["WAPE"] for m, s in val_scores.items()}).T.loc[MODELS].drop(columns="All")
bias_wh = pd.DataFrame({m: s["bias"] for m, s in val_scores.items()}).T.loc[MODELS].drop(columns="All")
print("Validation WAPE % by warehouse:")
display((wape_wh * 100).round(2))
print("Validation bias % by warehouse (positive = over-forecast):")
display((bias_wh * 100).round(2))

Leakage check passed: no baseline sees the day it forecasts
Validation, all warehouses together (sorted by WAPE):


,MAE units,WAPE %,bias %
median8,647.3,8.49,-2.12
mean8,648.0,8.50,-1.13
median4,649.6,8.52,-2.63
mean4,683.7,8.96,-1.52
mean4_level,764.6,10.02,-0.78
mean8_level,795.7,10.43,-0.34
seasonal_naive,920.5,12.07,-0.36
naive,1405.8,18.43,-0.18


Validation WAPE % by warehouse:


,Cuenca,Guayaquil,Quito,Santo Domingo
naive,15.90,18.67,18.43,21.92
seasonal_naive,10.87,12.82,12.02,11.68
mean4,7.74,9.99,8.76,8.97
mean8,7.29,9.53,8.25,8.75
median4,7.54,9.98,8.02,8.98
median8,7.02,10.15,7.93,9.47
mean4_level,9.48,10.12,10.18,9.18
mean8_level,9.54,10.37,10.70,9.84


Validation bias % by warehouse (positive = over-forecast):


,Cuenca,Guayaquil,Quito,Santo Domingo
naive,-0.10,-0.06,-0.25,-0.18
seasonal_naive,-0.72,-0.38,-0.22,-0.92
mean4,-1.82,-2.86,-0.67,-3.40
mean8,-1.54,-5.10,1.09,-4.62
median4,-2.52,-4.13,-1.76,-4.77
median8,-3.14,-6.01,0.29,-6.54
mean4_level,-0.80,-1.71,-0.25,-1.80
mean8_level,-0.45,-3.90,1.55,-2.97


### How to read it
(Fill in together after running.)

### What it means
(Fill in together after running.)

### What's next and why
(Fill in together after running.)

In [30]:
# 5d. Choose the best baseline: lowest overall WAPE on VALIDATION (test is not touched here)
best_model = overall["WAPE"].idxmin()
print(f"Best baseline on validation: {best_model}  (WAPE {overall.loc[best_model, 'WAPE']:.2%})")
print(f"Reference, plain seasonal naive:  WAPE {overall.loc['seasonal_naive', 'WAPE']:.2%}")
gain = 1 - overall.loc[best_model, "WAPE"] / overall.loc["seasonal_naive", "WAPE"]
print(f"Relative improvement over seasonal naive: {gain:.1%}")

# Does the level adjustment help? Compare each plain mean with its _level version.
for n in [4, 8]:
    plain, adj = overall.loc[f"mean{n}", "WAPE"], overall.loc[f"mean{n}_level", "WAPE"]
    print(f"mean{n}: {plain:.2%}  ->  with level adjustment: {adj:.2%}  ({'helps' if adj < plain else 'does not help'})")

print("Best vs seasonal naive by warehouse (validation):")
display(pretty(pd.concat({best_model: val_scores[best_model], "seasonal_naive": val_scores["seasonal_naive"]}, names=["model", "scope"])))

Best baseline on validation: median8  (WAPE 8.49%)
Reference, plain seasonal naive:  WAPE 12.07%
Relative improvement over seasonal naive: 29.7%
mean4: 8.96%  ->  with level adjustment: 10.02%  (does not help)
mean8: 8.50%  ->  with level adjustment: 10.43%  (does not help)
Best vs seasonal naive by warehouse (validation):


days  MAE units  WAPE %  bias %
model          scope                                         
median8        All             224      647.3    8.49   -2.12
               Cuenca           56      242.0    7.02   -3.14
               Guayaquil        56      776.1   10.15   -6.01
               Quito            56     1383.6    7.93    0.29
               Santo Domingo    56      187.5    9.47   -6.54
seasonal_naive All             224      920.5   12.07   -0.36
               Cuenca           56      374.6   10.87   -0.72
               Guayaquil        56      979.8   12.82   -0.38
               Quito            56     2096.2   12.02   -0.22
               Santo Domingo    56      231.3   11.68   -0.92

### How to read it
(Fill in together after running.)

### What it means
(Fill in together after running.)

### What's next and why
(Fill in together after running.)

In [31]:
# 5e. Score on TEST: once, only the chosen baseline and the plain seasonal naive. No tuning after this.
test_models = list(dict.fromkeys([best_model, "seasonal_naive"]))
test_fc = pd.concat([forecast_table(m, "test") for m in test_models], ignore_index=True)
test_scores = {m: score(g) for m, g in test_fc.groupby("model")}

for m in test_models:
    print(f"TEST, model = {m}")
    display(pretty(test_scores[m]))

# Save every forecast (validation: all candidates; test: the two models above) in TOTAL units.
results_b = pd.concat([val_fc, test_fc], ignore_index=True)
results_b["forecast"] = results_b["forecast"].round(3)
results_b["actual"] = results_b["actual"].round(3)
results_b.to_csv(PROCESSED / "results_b.csv", index=False)
print(f"Saved {len(results_b):,} rows to {PROCESSED / 'results_b.csv'}")

TEST, model = median8


,days,MAE units,WAPE %,bias %
All,224,510.8,6.92,1.77
Cuenca,56,224.1,6.72,2.35
Guayaquil,56,535.0,6.84,-2.91
Quito,56,1161.1,7.12,4.47
Santo Domingo,56,123.0,5.96,-2.81


TEST, model = seasonal_naive


,days,MAE units,WAPE %,bias %
All,224,575.3,7.80,0.91
Cuenca,56,244.8,7.35,1.48
Guayaquil,56,737.1,9.42,-1.00
Quito,56,1167.2,7.16,1.85
Santo Domingo,56,152.0,7.36,-0.18


Saved 2,240 rows to ..\data\processed\results_b.csv


### How to read it
(Fill in together after running.)

### What it means
(Fill in together after running.)

### What's next and why
(Fill in together after running.)

In [32]:
# 5f. Chart: actual vs the chosen baseline on the test split, one panel per warehouse
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warehouses = ["Quito", "Guayaquil", "Cuenca", "Santo Domingo"]
shown = test_fc[test_fc["model"] == best_model]

fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.05, subplot_titles=warehouses)
for i, wh in enumerate(warehouses, start=1):
    g = shown[shown["warehouse"] == wh].sort_values("date")
    fig.add_trace(go.Scatter(x=g["date"], y=g["actual"], mode="lines", name="Actual",
                             line=dict(color="#333333", width=1.5), legendgroup="actual",
                             showlegend=(i == 1)), row=i, col=1)
    fig.add_trace(go.Scatter(x=g["date"], y=g["forecast"], mode="lines", name=f"Forecast ({best_model})",
                             line=dict(color="#4f7a9c", width=1.5, dash="dot"), legendgroup="fc",
                             showlegend=(i == 1)), row=i, col=1)
    fig.update_yaxes(title_text="Units", row=i, col=1)
    # Dotted grey line: 2017-08-11, the one national day off in the test window.
    fig.add_shape(type="line", x0="2017-08-11", x1="2017-08-11", y0=0, y1=1, yref="y domain" if i == 1 else f"y{i} domain",
                  xref="x" if i == 1 else f"x{i}", line=dict(color="#aaaaaa", width=1, dash="dot"))
fig.update_xaxes(title_text="Date (test split)", row=4, col=1)
fig.update_layout(title=f"Test split: actual vs {best_model} forecast, total units per warehouse-day "
                        "(grey dotted line = 11 Aug national day off)",
                  template="plotly_white", height=800, legend_title_text="")
fig.show()

### How to read it
(Fill in together after running.)

### What it means
(Fill in together after running.)

### What's next and why
(Fill in together after running.)

## Step 6. A better model: gradient boosting

### What we are looking for
A model that predicts clearly better than the best baseline, and an understanding of what it learned. This is also where we learn what gradient boosting is.

### How we got here (the process)
1. **Features that only use the past (6b).** Units per open store 1, 7, 14, 21 and 28 days earlier; averages of the last 7 and 28 days and of the last 4 same weekdays; weekday, month and day of month (paydays); national holiday today and tomorrow; the extra shopping days around Christmas; and the warehouse. Everything stops at day d-1. 112 rows at the start of each warehouse lack a full 28-day history and are dropped.
2. **Left out on purpose:** items on promotion (nothing shows it is known a day ahead; using it would quietly assume so), local holidays (they apply to cities, not to our assumed warehouse groups) and Mother's Day-type "Event" days.
3. **Three candidates, 36 settings in total, all tried on validation only (6c):** (a) ridge regression, a straight-line model with a penalty that keeps the numbers from getting extreme; (b) gradient boosting on units per open store; (c) the same boosting model made "scale-free": every lag and average is divided by the last 28-day average, the model predicts today relative to recent level (about 1.34 on a typical Sunday), and the answer is scaled back. We added (c) because a tree model cannot follow each warehouse's different and drifting level, and this lets all four warehouses share one pattern.
4. **What gradient boosting is:** a decision tree asks a chain of yes/no questions and ends in a number; one small tree is crude. Boosting builds many small trees in turn, each trained only on what the previous ones got wrong, and adds them up as small corrections.
5. **Choose on validation (6d), then refit and score on test, once (6e).** The winner is refit on train plus validation with its chosen settings and forecasts each test day from the real values of the days before it.
6. **What did it lean on (6f):** permutation importance, which shuffles one feature and measures how much the error rises.

**Assumptions and limits:** scores use the actual values of earlier days, so this is a one-day-ahead score; a longer horizon would be worse. Only 56 test days, none at year end or on Mother's Day, so test looks better than validation partly because validation contains two special days. Holiday effects are learned from very few examples. The best settings sit at the edge of the grid we tried. The forecast is one number per day, with no range yet.

In [33]:
# 6a. Imports, load the bread table, and build the calendar (holiday) flags
import warnings
from itertools import product
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROCESSED = Path("../data/processed")
bread = pd.read_csv(PROCESSED / "bread_daily.csv", parse_dates=["date"])
print(f"Rows: {len(bread):,}; splits: {bread['split'].value_counts().to_dict()}")

# National days off, known long in advance. A "transferred" holiday was moved to another date,
# so the original date was a normal day. "Additional" days are the extra shopping days around
# Christmas / New Year; I give them their own flag because they are not days off.
hol = pd.read_csv("../data/raw/store-sales-time-series-forecasting/holidays_events.csv", parse_dates=["date"])
nat = hol[(hol["locale"] == "National") & (~hol["transferred"])]
OFF_DAYS = set(nat.loc[nat["type"].isin(["Holiday", "Transfer", "Bridge"]), "date"])
EXTRA_DAYS = set(nat.loc[nat["type"] == "Additional", "date"])
print(f"National days off in the file: {len(OFF_DAYS)}; extra holiday-season days: {len(EXTRA_DAYS)}")

Rows: 3,815; splits: {'train': 3367, 'validation': 224, 'test': 224}
National days off in the file: 65; extra holiday-season days: 40


### How to read it
(Fill in together after running.)

### What it means
(Fill in together after running.)

### What's next and why
(Fill in together after running.)

In [34]:
# 6b. Build features that use ONLY the past (one-day-ahead: day d sees data up to d-1)
# Gap rule: put every warehouse on a complete daily calendar. Where a day is missing (Jan 1
# closures, Dec 25) fill it with the value 7 days earlier (same weekday), so "yesterday" always
# exists and the weekly rhythm is kept. Filled days are only used as history, never scored.
LAGS = [1, 7, 14, 21, 28]
pieces, n_filled = [], 0
for wh, grp in bread.groupby("warehouse"):
    calendar = pd.date_range(grp["date"].min(), grp["date"].max(), freq="D")
    y = grp.set_index("date")["units_per_open_store"].reindex(calendar)
    vals = y.to_numpy().copy()
    for i in np.flatnonzero(np.isnan(vals)):
        vals[i] = vals[i - 7]          # always a past value, already filled if it was missing too
        n_filled += 1
    y = pd.Series(vals, index=calendar)

    f = pd.DataFrame(index=calendar)
    for k in LAGS:
        f[f"lag{k}"] = y.shift(k)
    f["mean7"] = y.shift(1).rolling(7).mean()      # last 7 days
    f["mean28"] = y.shift(1).rolling(28).mean()    # last 28 days
    f["same_wd4"] = f[["lag7", "lag14", "lag21", "lag28"]].mean(axis=1)  # last 4 same weekdays
    f["warehouse"] = wh
    pieces.append(f.rename_axis("date").reset_index())
print(f"Missing days filled with the value 7 days earlier: {n_filled}")

feat = pd.concat(pieces, ignore_index=True)
data = bread.merge(feat, on=["date", "warehouse"], how="left", validate="one_to_one")

# Calendar features: known in advance for any future day.
data["weekday"] = data["date"].dt.dayofweek
data["month"] = data["date"].dt.month
data["day_of_month"] = data["date"].dt.day          # paydays are mid-month and month-end
data["is_holiday"] = data["date"].isin(OFF_DAYS).astype(int)
data["holiday_tomorrow"] = (data["date"] + pd.Timedelta(days=1)).isin(OFF_DAYS).astype(int)
data["extra_day"] = data["date"].isin(EXTRA_DAYS).astype(int)
data["wh_code"] = data["warehouse"].astype("category").cat.codes   # for the tree models

# The first 28 days of the data have no full history: drop those rows (no model is asked about them).
before = len(data)
data = data.dropna(subset=["lag28"]).reset_index(drop=True)
print(f"Dropped {before - len(data)} early rows with incomplete history; {len(data):,} rows remain")
# on_promotion is deliberately NOT a feature: it is the count of items on promotion on day d itself.
# Nothing here shows it is fixed a day ahead, and using it would quietly assume it is.
assert data[[f"lag{k}" for k in LAGS] + ["mean7", "mean28", "same_wd4"]].notna().all().all()
display(data[["date", "warehouse", "units_per_open_store", "lag1", "lag7", "mean7", "same_wd4", "weekday",
              "is_holiday", "holiday_tomorrow"]].tail(6))

Missing days filled with the value 7 days earlier: 14
Dropped 112 early rows with incomplete history; 3,703 rows remain


,date,warehouse,units_per_open_store,lag1,lag7,mean7,same_wd4,weekday,is_holiday,holiday_tomorrow
3697,2017-08-10,Santo Domingo,256.783500,279.349667,264.414167,345.127667,262.906667,3,0,1
3698,2017-08-11,Santo Domingo,271.531000,256.783500,288.704333,344.037571,274.251333,4,1,0
3699,2017-08-12,Santo Domingo,266.315500,271.531000,349.043833,341.584238,339.919750,5,0,0
3700,2017-08-13,Santo Domingo,500.237500,266.315500,504.508500,329.765905,493.643542,6,0,0
3701,2017-08-14,Santo Domingo,346.525000,500.237500,381.548000,329.155762,381.321875,0,0,0
3702,2017-08-15,Santo Domingo,360.072333,346.525000,348.325167,324.152476,345.641833,1,0,0


### How to read it
(Fill in together after running.)

### What it means
(Fill in together after running.)

### What's next and why
(Fill in together after running.)

In [35]:
# 6c. The scoring function (used for every model) and the three candidates
def score(df):
    """MAE, WAPE and bias on TOTAL units, overall and per warehouse. df needs warehouse, actual, forecast."""
    rows = {}
    for name, part in [("ALL", df)] + list(df.groupby("warehouse")):
        err = part["forecast"] - part["actual"]
        rows[name] = {"MAE": err.abs().mean(),
                      "WAPE": err.abs().sum() / part["actual"].abs().sum(),   # headline
                      "bias": err.sum() / part["actual"].sum(),               # + over-forecast, - stockout risk
                      "n": len(part)}
    return pd.DataFrame(rows).T

CAL = ["weekday", "month", "day_of_month", "is_holiday", "holiday_tomorrow", "extra_day"]
LAG_COLS = ["lag1", "lag7", "lag14", "lag21", "lag28", "mean7", "mean28", "same_wd4"]
TREE_COLS = LAG_COLS + CAL + ["wh_code"]
TREE_CAT = ["weekday", "month", "wh_code"]

def make_model(name, p):
    """Return a fresh, untrained model for candidate `name` with settings `p`."""
    if name == "a_ridge":
        cols = ColumnTransformer([
            ("num", StandardScaler(), LAG_COLS + ["day_of_month", "is_holiday", "holiday_tomorrow", "extra_day"]),
            ("cat", OneHotEncoder(handle_unknown="ignore"), ["weekday", "warehouse"])])
        return make_pipeline(cols, Ridge(alpha=p["alpha"]))
    return HistGradientBoostingRegressor(
        learning_rate=p["learning_rate"], max_leaf_nodes=p["max_leaf_nodes"], max_iter=p["max_iter"],
        loss=p["loss"], categorical_features=TREE_CAT, early_stopping=False, random_state=0)

def fit_predict(name, p, train_df, pred_df):
    """Train on train_df, forecast pred_df. Returns forecast in TOTAL units."""
    if name == "c_hgb_ratio":
        # Scale-free version: predict "today relative to the last 28 days" and rescale afterwards.
        def rel(d):
            r = d[TREE_COLS].copy()
            for c in LAG_COLS:
                r[c] = d[c] / d["mean28"]
            return r
        model = make_model(name, p).fit(rel(train_df), train_df["units_per_open_store"] / train_df["mean28"])
        per_store = model.predict(rel(pred_df)) * pred_df["mean28"].to_numpy()
    else:
        X = (lambda d: d[TREE_COLS]) if name == "b_hgb" else (lambda d: d)
        model = make_model(name, p).fit(X(train_df), train_df["units_per_open_store"])
        per_store = model.predict(X(pred_df))
    return np.clip(per_store, 0, None) * pred_df["open_stores"].to_numpy()

GRIDS = {
    "a_ridge": [{"alpha": a} for a in [0.1, 1, 10, 100]],
    "b_hgb": [dict(learning_rate=lr, max_leaf_nodes=nl, max_iter=it, loss=ls)
              for lr, nl, it, ls in product([0.05, 0.1], [8, 16], [150, 300], ["squared_error", "absolute_error"])],
}
GRIDS["c_hgb_ratio"] = GRIDS["b_hgb"]   # same settings grid as (b), so the comparison is fair

train = data[data["split"] == "train"]
val = data[data["split"] == "validation"]

# Reference only (not a candidate): "same weekday last week" = lag7 x open stores.
ref = val.assign(actual=val["sales"], forecast=val["lag7"] * val["open_stores"])
ref_score = score(ref).loc["ALL"]
print(f"Reference, same weekday last week: validation WAPE {ref_score['WAPE']:.3%}")

# Every setting of every candidate is trained on train only and scored on validation only.
tuning, val_forecasts = [], {}
for name, grid in GRIDS.items():
    for i, p in enumerate(grid):
        fc = fit_predict(name, p, train, val)
        scored = val.assign(actual=val["sales"], forecast=fc)
        s = score(scored).loc["ALL"]
        tuning.append({"candidate": name, "setting": i, "params": str(p), "val_WAPE": s["WAPE"],
                       "val_MAE": s["MAE"], "val_bias": s["bias"]})
        val_forecasts[(name, i)] = scored
tuning = pd.DataFrame(tuning)
print(f"Settings tried: {len(tuning)} across {tuning['candidate'].nunique()} candidates")
display(tuning.sort_values(["candidate", "val_WAPE"]).groupby("candidate").head(3).round(4))

Reference, same weekday last week: validation WAPE 12.069%
Settings tried: 36 across 3 candidates


,candidate,setting,params,val_WAPE,val_MAE,val_bias
0,a_ridge,0,{'alpha': 0.1},0.0799,609.7398,-0.0110
1,a_ridge,1,{'alpha': 1},0.0799,609.7459,-0.0109
2,a_ridge,2,{'alpha': 10},0.0800,609.8739,-0.0107
19,b_hgb,15,"{'learning_rate': 0.1, 'max_leaf_nodes': 16, '...",0.0663,505.5198,-0.0030
18,b_hgb,14,"{'learning_rate': 0.1, 'max_leaf_nodes': 16, '...",0.0677,516.6346,0.0053
14,b_hgb,10,"{'learning_rate': 0.1, 'max_leaf_nodes': 8, 'm...",0.0680,518.5709,0.0049
34,c_hgb_ratio,14,"{'learning_rate': 0.1, 'max_leaf_nodes': 16, '...",0.0588,448.2312,0.0003
32,c_hgb_ratio,12,"{'learning_rate': 0.1, 'max_leaf_nodes': 16, '...",0.0590,450.2122,-0.0001
27,c_hgb_ratio,7,"{'learning_rate': 0.05, 'max_leaf_nodes': 16, ...",0.0596,454.4813,-0.0042


### How to read it
(Fill in together after running.)

### What it means
(Fill in together after running.)

### What's next and why
(Fill in together after running.)

In [36]:
# 6d. Pick the winner by validation WAPE (best setting per candidate, then best candidate)
best_rows = tuning.loc[tuning.groupby("candidate")["val_WAPE"].idxmin()].sort_values("val_WAPE")
display(best_rows[["candidate", "params", "val_WAPE", "val_MAE", "val_bias"]].round(4).reset_index(drop=True))

winner = best_rows.iloc[0]
BEST_NAME = winner["candidate"]
BEST_PARAMS = GRIDS[BEST_NAME][int(winner["setting"])]
print(f"Chosen: {BEST_NAME} with {BEST_PARAMS}  (validation WAPE {winner['val_WAPE']:.3%})")

# Validation forecasts of each candidate's best setting, kept for the results file.
val_out = pd.concat([
    val_forecasts[(r["candidate"], int(r["setting"]))].assign(model=r["candidate"])
    for _, r in best_rows.iterrows()])
val_out = val_out[["date", "warehouse", "split", "model", "actual", "forecast"]]

by_wh = pd.concat({m: score(g)["WAPE"] for m, g in val_out.groupby("model")}, axis=1)
print("Validation WAPE by warehouse for each candidate's best setting:")
display(by_wh.round(4))

,candidate,params,val_WAPE,val_MAE,val_bias
0,c_hgb_ratio,"{'learning_rate': 0.1, 'max_leaf_nodes': 16, '...",0.0588,448.2312,0.0003
1,b_hgb,"{'learning_rate': 0.1, 'max_leaf_nodes': 16, '...",0.0663,505.5198,-0.0030
2,a_ridge,{'alpha': 0.1},0.0799,609.7398,-0.0110


Chosen: c_hgb_ratio with {'learning_rate': 0.1, 'max_leaf_nodes': 16, 'max_iter': 300, 'loss': 'squared_error'}  (validation WAPE 5.877%)
Validation WAPE by warehouse for each candidate's best setting:


,a_ridge,b_hgb,c_hgb_ratio
ALL,0.0799,0.0663,0.0588
Cuenca,0.0699,0.0629,0.0641
Guayaquil,0.0879,0.0791,0.0709
Quito,0.0777,0.0600,0.0521
Santo Domingo,0.0860,0.0779,0.0615


### How to read it
(Fill in together after running.)

### What it means
(Fill in together after running.)

### What's next and why
(Fill in together after running.)

In [37]:
# 6e. Refit the chosen model on train + validation, then score on TEST - exactly once
# Test is looked at here and nowhere else; nothing is tuned after this cell.
assert "test" not in val_out["split"].values, "test rows leaked into model selection"
train_val = data[data["split"].isin(["train", "validation"])]
test = data[data["split"] == "test"]

# One-day-ahead with actual history: each test day's lags are the real values of the days before it.
test_fc = fit_predict(BEST_NAME, BEST_PARAMS, train_val, test)
test_out = test.assign(actual=test["sales"], forecast=test_fc, model=BEST_NAME)
test_out = test_out[["date", "warehouse", "split", "model", "actual", "forecast"]]

test_score = score(test_out)
print(f"TEST ({BEST_NAME}, scored once): WAPE {test_score.loc['ALL', 'WAPE']:.3%}, "
      f"MAE {test_score.loc['ALL', 'MAE']:.0f} units, bias {test_score.loc['ALL', 'bias']:+.2%}")
display(test_score.round(4))

results = pd.concat([val_out, test_out], ignore_index=True)
results.to_csv(PROCESSED / "results_c.csv", index=False)
print(f"Saved {len(results):,} rows to {PROCESSED / 'results_c.csv'}")

TEST (c_hgb_ratio, scored once): WAPE 3.741%, MAE 276 units, bias +0.49%


,MAE,WAPE,bias,n
ALL,276.0457,0.0374,0.0049,224.0
Cuenca,167.4402,0.0502,0.0081,56.0
Guayaquil,325.5159,0.0416,-0.0083,56.0
Quito,493.0920,0.0302,0.0102,56.0
Santo Domingo,118.1346,0.0572,0.0078,56.0


Saved 896 rows to ..\data\processed\results_c.csv


### How to read it
(Fill in together after running.)

### What it means
(Fill in together after running.)

### What's next and why
(Fill in together after running.)

In [38]:
# 6f. What did the model lean on? Permutation importance on VALIDATION
# Idea: shuffle one feature's values, so it carries no information, and see how much worse the
# forecast gets. A big rise in WAPE means the model relied on that feature. The model here is the
# chosen one trained on train only (so validation is genuinely unseen by it).
rng = np.random.default_rng(0)
# Train once, then re-predict on shuffled copies of validation.
if BEST_NAME == "c_hgb_ratio":
    def rel(d):
        r = d[TREE_COLS].copy()
        for c in LAG_COLS:
            r[c] = d[c] / d["mean28"]
        return r
    m = make_model(BEST_NAME, BEST_PARAMS).fit(rel(train), train["units_per_open_store"] / train["mean28"])
    fit_predict_fixed = lambda d: np.clip(m.predict(rel(d)) * d["mean28"].to_numpy(), 0, None) * d["open_stores"].to_numpy()
elif BEST_NAME == "b_hgb":
    m = make_model(BEST_NAME, BEST_PARAMS).fit(train[TREE_COLS], train["units_per_open_store"])
    fit_predict_fixed = lambda d: np.clip(m.predict(d[TREE_COLS]), 0, None) * d["open_stores"].to_numpy()
else:
    m = make_model(BEST_NAME, BEST_PARAMS).fit(train, train["units_per_open_store"])
    fit_predict_fixed = lambda d: np.clip(m.predict(d), 0, None) * d["open_stores"].to_numpy()

def val_wape(d):
    return np.abs(fit_predict_fixed(d) - d["sales"]).sum() / d["sales"].sum()

base = val_wape(val)
features = (["warehouse"] if BEST_NAME == "a_ridge" else ["wh_code"]) + LAG_COLS + CAL
rows = []
for f in features:
    worse = []
    for _ in range(20):
        shuffled = val.copy()
        shuffled[f] = rng.permutation(shuffled[f].to_numpy())
        worse.append(val_wape(shuffled) - base)
    rows.append({"feature": "warehouse" if f == "wh_code" else f, "WAPE_increase_pts": np.mean(worse) * 100})
importance = pd.DataFrame(rows).sort_values("WAPE_increase_pts", ascending=False).reset_index(drop=True)
print(f"Validation WAPE with nothing shuffled: {base:.3%}")
display(importance.round(3))

Validation WAPE with nothing shuffled: 5.877%


,feature,WAPE_increase_pts
0,mean28,11.834
1,weekday,6.482
2,lag1,3.085
3,same_wd4,2.349
4,day_of_month,2.015
5,warehouse,0.712
6,is_holiday,0.646
7,mean7,0.605
8,lag14,0.584
9,month,0.529


### How to read it
(Fill in together after running.)

### What it means
(Fill in together after running.)

### What's next and why
(Fill in together after running.)

In [39]:
# 6g. Chart: test period, actual vs forecast, one panel per warehouse
WAREHOUSES = ["Quito", "Guayaquil", "Cuenca", "Santo Domingo"]
fig = make_subplots(rows=2, cols=2, subplot_titles=WAREHOUSES, shared_xaxes=True, vertical_spacing=0.12)
for i, wh in enumerate(WAREHOUSES):
    r, c = i // 2 + 1, i % 2 + 1
    part = test_out[test_out["warehouse"] == wh].sort_values("date")
    fig.add_trace(go.Scatter(x=part["date"], y=part["actual"], mode="lines", name="Actual",
                             line=dict(color="#444444", width=1.5), showlegend=(i == 0)), row=r, col=c)
    fig.add_trace(go.Scatter(x=part["date"], y=part["forecast"], mode="lines", name="Forecast (day before)",
                             line=dict(color="#4f7a9c", width=1.5, dash="dot"), showlegend=(i == 0)), row=r, col=c)
    fig.update_yaxes(title_text="Units per day" if c == 1 else None, row=r, col=c)
    fig.update_xaxes(title_text="Date" if r == 2 else None, row=r, col=c)
fig.update_layout(title=f"Test period: actual vs one-day-ahead forecast ({BEST_NAME}), total bread units",
                  template="plotly_white", height=600, legend_title_text="")
fig.show()

### How to read it
(Fill in together after running.)

### What it means
(Fill in together after running.)

### What's next and why
(Fill in together after running.)

## Step 7. Compare the forecasts on the test weeks

### What we are looking for
A fair, side-by-side answer to: does the machine-learning model predict better than the simple rules, by how much, and where?

### How we got here (the process)
1. **Use the saved forecasts only.** Step 5 saved `results_b.csv` and Step 6 saved `results_c.csv`. This step does no new modelling and tunes nothing, so it cannot leak anything from the test weeks back into a choice.
2. **Same scoring for everyone (7a).** WAPE, MAE and bias on total units per warehouse-day, for the seasonal naive reference, the chosen baseline (median of the last 8 same weekdays) and the chosen model (scale-free boosting), on the 224 warehouse-days of the test weeks.
3. **By warehouse (7b) and by weekday (7c)** to see where each forecast is weak.

**Assumptions and limits:** 56 test days from one mid-year stretch. The result says nothing about December. These are one-day-ahead forecasts using actual earlier values.

In [40]:
# 7a. Side by side on the TEST weeks: the reference, the best simple baseline and the machine-learning model.
# Every number comes from the saved forecasts (results_b.csv and results_c.csv), scored the same way.
res_b = pd.read_csv(PROCESSED / "results_b.csv", parse_dates=["date"])
res_c = pd.read_csv(PROCESSED / "results_c.csv", parse_dates=["date"])
res = pd.concat([res_b, res_c], ignore_index=True)

def summarize(g):
    err = g["forecast"] - g["actual"]
    return pd.Series({
        "WAPE_%": 100 * err.abs().sum() / g["actual"].sum(),
        "MAE_units": err.abs().mean(),
        "bias_%": 100 * err.sum() / g["actual"].sum(),
        "warehouse_days": len(g),
    })

test_res = res[res["split"] == "test"]
order = ["seasonal_naive", "median8", "c_hgb_ratio"]
compare_test = test_res.groupby("model").apply(summarize, include_groups=False).loc[order].round(2)
display(compare_test)

base = compare_test.loc["median8", "WAPE_%"]
best = compare_test.loc["c_hgb_ratio", "WAPE_%"]
print(f"The machine-learning model misses by {best:.2f}% of sales; the best simple baseline misses by {base:.2f}%.")
print(f"That is {100 * (base - best) / base:.0f}% less error than the baseline.")

,WAPE_%,MAE_units,bias_%,warehouse_days
model,,,,
seasonal_naive,7.80,575.28,0.91,224.0
median8,6.92,510.80,1.77,224.0
c_hgb_ratio,3.74,276.05,0.49,224.0


The machine-learning model misses by 3.74% of sales; the best simple baseline misses by 6.92%.
That is 46% less error than the baseline.


### How to read it
(Fill in together after running.)

### What it means
(Fill in together after running.)

### What's next and why
(Fill in together after running.)

In [41]:
# 7b. The same comparison for each warehouse, as a chart (lower is better).
import plotly.express as px

by_wh = (
    test_res[test_res["model"].isin(order)]
    .groupby(["warehouse", "model"]).apply(summarize, include_groups=False)
    .reset_index()
)
by_wh["WAPE_%"] = by_wh["WAPE_%"].round(2)
display(by_wh.pivot(index="warehouse", columns="model", values="WAPE_%")[order])

fig = px.bar(by_wh, x="warehouse", y="WAPE_%", color="model", barmode="group",
             category_orders={"model": order}, title="Forecast error on the 8 test weeks, by warehouse (WAPE %, lower is better)",
             labels={"WAPE_%": "WAPE (%)", "warehouse": "", "model": "Forecast"})
fig.update_layout(height=450)
fig.show()

model,seasonal_naive,median8,c_hgb_ratio
warehouse,,,
Cuenca,7.35,6.72,5.02
Guayaquil,9.42,6.84,4.16
Quito,7.16,7.12,3.02
Santo Domingo,7.36,5.96,5.72


### How to read it
(Fill in together after running.)

### What it means
(Fill in together after running.)

### What's next and why
(Fill in together after running.)

In [42]:
# 7c. Where do the forecasts miss? Error by weekday on the test weeks (WAPE %, lower is better).
wd = test_res[test_res["model"].isin(order)].assign(weekday=lambda d: d["date"].dt.day_name())
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
by_weekday = (
    wd.groupby(["weekday", "model"]).apply(summarize, include_groups=False)["WAPE_%"].unstack("model")[order]
    .loc[weekday_order].round(2)
)
display(by_weekday)

model,seasonal_naive,median8,c_hgb_ratio
weekday,,,
Monday,6.36,5.30,3.20
Tuesday,11.29,7.23,3.99
Wednesday,7.20,5.79,2.58
Thursday,5.05,4.08,2.66
Friday,4.91,5.48,4.07
Saturday,9.40,9.76,5.08
Sunday,9.22,9.11,4.10


### How to read it
(Fill in together after running.)

### What it means
(Fill in together after running.)

### What's next and why
(Fill in together after running.)

### Milestone 3 conclusion
(Write together after reading every step: what we built, what we found, the limits, and what comes next.)